In [18]:
import pathlib
import numpy
import polars
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig
from data_index.analysis.tables import IMOS_DATA_LIVE_TABLE
from data_index.analysis.datasets import (
    DATASET,
    get_dataset_objects_df,
    get_dataset_arrow_dataset,
)

In [2]:
table = IMOS_DATA_LIVE_TABLE.load()

In [3]:
df = table.scan(
    selected_fields=("bucket", "key", "size", "facility", "last_modified_date",),
    row_filter=(
        "facility == 'ANFOG'"
    ),
).to_polars()

In [4]:
dataset: DATASET = "ocean_glider_delayed_qc"

In [5]:
dataset_df = (
    get_dataset_objects_df(
        df=df,
        dataset=dataset,
    )
)
dataset_df

bucket,key,size,last_modified_date,facility,uri,filename,start_date,end_date
str,str,i64,datetime[μs],str,str,str,date,date
"""imos-data""","""IMOS/ANFOG/seaglider/Bicheno20…",39552768,2016-03-30 23:04:10,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/seag…","""IMOS_ANFOG_BCEOPSTUV_20090422T…",2009-04-22,2009-06-22
"""imos-data""","""IMOS/ANFOG/seaglider/CoralSea2…",48063675,2016-03-31 06:16:25,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/seag…","""IMOS_ANFOG_BCEOPSTUV_20130618T…",2013-06-18,2013-09-29
"""imos-data""","""IMOS/ANFOG/seaglider/GAB201409…",14100739,2016-03-31 09:36:25,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/seag…","""IMOS_ANFOG_BCEOPSTUV_20140916T…",2014-09-16,2014-10-12
"""imos-data""","""IMOS/ANFOG/seaglider/GAB201603…",48109617,2016-06-14 01:18:02,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/seag…","""IMOS_ANFOG_BCEOPSTUV_20160310T…",2016-03-10,2016-05-25
"""imos-data""","""IMOS/ANFOG/seaglider/Leeuwin20…",45510496,2016-04-01 03:42:12,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/seag…","""IMOS_ANFOG_BCEOPSTUV_20131017T…",2013-10-17,2014-01-06
…,…,…,…,…,…,…,…,…
"""imos-data""","""IMOS/ANFOG/slocum_glider/TasEa…",99068286,2024-12-05 06:43:00,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/sloc…","""IMOS_ANFOG_BCEOPSTUV_20241030T…",2024-10-30,2024-11-24
"""imos-data""","""IMOS/ANFOG/slocum_glider/TwoRo…",3096988,2024-02-06 06:48:18,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/sloc…","""IMOS_ANFOG_BCEOPSTUV_20220916T…",2022-09-16,2022-09-17
"""imos-data""","""IMOS/ANFOG/slocum_glider/TwoRo…",122779831,2024-03-07 05:28:00,"""ANFOG""","""s3://imos-data/IMOS/ANFOG/sloc…","""IMOS_ANFOG_BCEOPSTUV_20240115T…",2024-01-15,2024-02-15


In [7]:
ds = get_dataset_arrow_dataset(dataset=dataset,)

In [27]:
nc_filenames = polars.DataFrame(data={"filename": dataset_df["key"].str.split("/").list.last().unique().sort()})
parquet_filenames = polars.DataFrame(data={
    "filename": [pathlib.Path(fragment.path).name for fragment in ds.get_fragments()]
}).unique()

In [30]:
len(nc_filenames) == len(parquet_filenames)

True